
# MCP (model context protocol)

MCP is a standardized way for applications to provide AI models with richer context about their environment, user preferences, and conversation history.
An Agent works with one or more LLMs and multiple tools (serpapi, yfinance, etc..)  


## MCP Components
* MCP server acts as the bridge between the model and external tools. It provides a structured list of available functions (like APIs, databases, or internal services), along with the input and output formats.


* MCP Host/Application: The platform or AI assistant that uses MCP to gain data (e.g. a chatbot interface, an IDE with an AI assistant, or a custom agent framework).

* MCP Client (Agent): The part of the AI system that communicates with MCP servers to send queries and receive results.


* Data/Tool Resources: The actual endpoints the MCP server accesses — these can be local files, databases, enterprise systems, cloud services, etc. The MCP server is configured with the credentials and logic to communicate with these resources.


###  Agent Context

MCP helps AI agents manage different types of context efficiently, enabling better coordination, memory handling and secure interactions across tasks. Agents typically handle three types of context:

* Ephemeral Context: Temporary data from the current interaction, used only during the task.
* Session Context: Short term information that persists across multiple steps in a workflow.
* Long-Term Memory: Stored data that the agent can reuse over time for better personalization and continuity.


### Think of MCP as a “USB-C port” for AI applications, providing a standardized two-way interface between AI and the world of data and software tools.






## Create and test sample weather mcp server

### Tools that are exposed by mcp server: get_weather, get_forecast



In [ ]:

#!/usr/bin/env python3
import json
import subprocess
import sys

def test_mcp_server_simple():
    """Simple synchronous test"""

    # Use the same Python interpreter that's running this script
    python_exe = sys.executable

    # Test commands
    tests = [
        {
            "name": "List Tools",
            "request": {"jsonrpc":"2.0","id":1,"method":"tools/list"}
        },
        {
            "name": "Get Weather",
            "request": {"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"get_weather","arguments":{"city":"London"}}}
        },
        {
            "name": "Get Forecast",
            "request": {"jsonrpc":"2.0","id":3,"method":"tools/call","params":{"name":"get_forecast","arguments":{"city":"Tokyo","days":3}}}
        }
    ]

    for test in tests:
        print(f"\n🧪 Testing: {test['name']}")
        print("-" * 40)

        # Use sys.executable instead of 'python'
        process = subprocess.Popen(
            [python_exe, '\weather_mcp_server.py'],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        # Send request and close stdin
        request_json = json.dumps(test['request'])
        stdout, stderr = process.communicate(input=request_json + '\n')

        # Print server startup message
        if stderr:
            print(f"Server: {stderr.strip()}")

        # Parse and display response
        if stdout.strip():
            try:
                response = json.loads(stdout.strip())
                if 'result' in response:
                    print("✅ Success!")
                    if test['name'] == 'List Tools':
                        tools = response['result']['tools']
                        for tool in tools:
                            print(f"   - {tool['name']}: {tool['description']}")
                    else:
                        content = response['result']['content'][0]['text']
                        data = json.loads(content)
                        print(f"   Response: {json.dumps(data, indent=2)}")
                else:
                    print(f"❌ Error: {response.get('error', 'Unknown error')}")
            except json.JSONDecodeError:
                print(f"❌ Invalid JSON response: {stdout}")
        else:
            print("❌ No response received")

if __name__ == "__main__":
    test_mcp_server_simple()

In [ ]:
import asyncio
import json
import sys
import os
import random
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

class WeatherMCPServer:
    def __init__(self):
        self.tools = [
            {
                "name": "get_weather",
                "description": "Get current weather information for a city",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "city": {
                            "type": "string",
                            "description": "City name to get weather for"
                        },
                        "units": {
                            "type": "string",
                            "enum": ["metric", "imperial", "kelvin"],
                            "description": "Temperature units",
                            "default": "metric"
                        }
                    },
                    "required": ["city"]
                }
            },
            {
                "name": "get_forecast",
                "description": "Get weather forecast for a city",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "city": {
                            "type": "string",
                            "description": "City name to get forecast for"
                        },
                        "days": {
                            "type": "integer",
                            "description": "Number of days (1-5)",
                            "minimum": 1,
                            "maximum": 5,
                            "default": 3
                        }
                    },
                    "required": ["city"]
                }
            }
        ]

    def get_weather(self, city: str, units: str = "metric") -> Dict[str, Any]:
        """Get current weather for a city"""
        api_key = os.getenv('OPENWEATHERMAP_API_KEY', 'xxxxx')

        if api_key == 'demo_key':
            # Return mock data for demo
            return {
                "content": [
                    {
                        "type": "text",
                        "text": json.dumps({
                            "city": city,
                            "temperature": "22°C",
                            "condition": "Partly cloudy",
                            "humidity": "65%",
                            "wind_speed": "10 km/h",
                            "note": "This is mock data. Add OPENWEATHER_API_KEY to .env for real data."
                        }, indent=2)
                    }
                ]
            }

        try:
            url = f"http://api.openweathermap.org/data/2.5/weather"
            params = {
                'q': city,
                'appid': api_key,
                #'units': units
            }

            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            unit_symbol = '°C' if units == 'metric' else ('°F' if units == 'imperial' else 'K')
            speed_unit = 'm/s' if units == 'metric' else 'mph'

            weather_data = {
                "city": data['name'],
                "country": data['sys']['country'],
                "temperature": f"{data['main']['temp']}{unit_symbol}",
                "condition": data['weather'][0]['description'],
                "humidity": f"{data['main']['humidity']}%",
                "wind_speed": f"{data['wind']['speed']} {speed_unit}",
                "pressure": f"{data['main']['pressure']} hPa"
            }

            return {
                "content": [
                    {
                        "type": "text",
                        "text": json.dumps(weather_data, indent=2)
                    }
                ]
            }

        except requests.exceptions.RequestException as e:
            return {
                "content": [
                    {
                        "type": "text",
                        "text": f"Error fetching weather data: {str(e)}"
                    }
                ]
            }

    def get_forecast(self, city: str, days: int = 3) -> Dict[str, Any]:
        """Get weather forecast for a city"""
        # Generate mock forecast data for demo
        forecasts = []
        conditions = ['Sunny', 'Cloudy', 'Rainy', 'Partly Cloudy', 'Snowy']

        for i in range(days):
            forecast_date = datetime.now() + timedelta(days=i)
            forecasts.append({
                "date": forecast_date.strftime("%Y-%m-%d"),
                "high": random.randint(15, 30),
                "low": random.randint(5, 15),
                "condition": random.choice(conditions)
            })

        forecast_data = {
            "city": city,
            "forecast": forecasts,
            "note": "This is mock forecast data for demonstration."
        }

        return {
            "content": [
                {
                    "type": "text",
                    "text": json.dumps(forecast_data, indent=2)
                }
            ]
        }

    async def handle_request(self, request: Dict[str, Any]) -> Dict[str, Any]:
        """Handle incoming JSON-RPC request"""
        method = request.get('method')
        params = request.get('params', {})
        request_id = request.get('id')

        try:
            if method == 'tools/list':
                result = {"tools": self.tools}

            elif method == 'tools/call':
                tool_name = params.get('name')
                arguments = params.get('arguments', {})

                if tool_name == 'get_weather':
                    result = self.get_weather(
                        arguments.get('city'),
                        arguments.get('units', 'metric')
                    )
                elif tool_name == 'get_forecast':
                    result = self.get_forecast(
                        arguments.get('city'),
                        arguments.get('days', 3)
                    )
                else:
                    raise ValueError(f"Unknown tool: {tool_name}")
            elif method == 'initialize':
                result = {
                    "protocolVersion": "2025-03-26",
                    "serverInfo": {"name": "Weather MCP (Py)", "version": "0.1.0"},
                    "capabilities": {
                        "tools": {}  # you only expose tools; add prompts/resources if you later support them
                    }
                }
                # Send the required notification after responding
                # NOTE: We print the notification here; it's a JSON-RPC *notification* (no "id")
                print(json.dumps({
                    "jsonrpc": "2.0",
                    "method": "notifications/initialized",
                    "params": {}
                }), flush=True)
            else:
                raise ValueError(f"Unknown method: {method}")

            return {
                "jsonrpc": "2.0",
                "id": request_id,
                "result": result
            }

        except Exception as e:
            return {
                "jsonrpc": "2.0",
                "id": request_id,
                "error": {
                    "code": -32000,
                    "message": str(e)
                }
            }

    async def run(self):
        """Run the MCP server"""
        print("Weather MCP Server (Python) running on stdio", file=sys.stderr)

        try:
            while True:
                # Read line from stdin with better error handling
                try:
                    line = await asyncio.get_event_loop().run_in_executor(
                        None, sys.stdin.readline
                    )

                    # Check for EOF or empty input
                    if not line or line.strip() == "":
                        break

                    # Parse JSON-RPC request
                    request = json.loads(line.strip())

                    # Handle request
                    response = await self.handle_request(request)

                    # Send response
                    print(json.dumps(response), flush=True)

                except json.JSONDecodeError as e:
                    # Only send error if we actually got input
                    if line and line.strip():
                        error_response = {
                            "jsonrpc": "2.0",
                            "id": None,
                            "error": {
                                "code": -32700,
                                "message": f"Parse error: {str(e)}"
                            }
                        }
                        print(json.dumps(error_response), flush=True)

                except EOFError:
                    break

        except KeyboardInterrupt:
            pass
        except Exception as e:
            print(f"Server error: {e}", file=sys.stderr)

async def main():
    server = WeatherMCPServer()
    await server.run()

if __name__ == "__main__":
  asyncio.run(main())

In [ ]:
Output:

Testing: List Tools
----------------------------------------
Server: Weather MCP Server (Python) running on stdio
✅ Success!
   - get_weather: Get current weather information for a city
   - get_forecast: Get weather forecast for a city

🧪 Testing: Get Weather
----------------------------------------
Server: Weather MCP Server (Python) running on stdio
✅ Success!
   Response: {
  "city": "London",
  "country": "GB",
  "temperature": "285.01\u00b0C",
  "condition": "clear sky",
  "humidity": "45%",
  "wind_speed": "2.86 m/s",
  "pressure": "1025 hPa"
}

🧪 Testing: Get Forecast
----------------------------------------
Server: Weather MCP Server (Python) running on stdio
✅ Success!
   Response: {
  "city": "Tokyo",
  "forecast": [
    {
      "date": "2026-04-06",
      "high": 22,
      "low": 8,
      "condition": "Snowy"
    },
    {
      "date": "2026-04-07",
      "high": 20,
      "low": 7,
      "condition": "Rainy"
    },
    {
      "date": "2026-04-08",
      "high": 24,
      "low": 12,
      "condition": "Cloudy"
    }
  ],
  "note": "This is mock forecast data."
}

In [ ]:
Output:

Testing: List Tools
----------------------------------------
Server: Weather MCP Server (Python) running on stdio
✅ Success!
   - get_weather: Get current weather information for a city
   - get_forecast: Get weather forecast for a city

🧪 Testing: Get Weather
----------------------------------------
Server: Weather MCP Server (Python) running on stdio
✅ Success!
   Response: {
  "city": "Hyderabad",
  "country": "IN",
  "temperature": "301.21\u00b0C",
  "condition": "clear sky",
  "humidity": "22%",
  "wind_speed": "3.09 m/s",
  "pressure": "1010 hPa"
}

🧪 Testing: Get Forecast
----------------------------------------
Server: Weather MCP Server (Python) running on stdio
✅ Success!
   Response: {
  "city": "Chicago",
  "forecast": [
    {
      "date": "2026-04-06",
      "high": 28,
      "low": 15,
      "condition": "Cloudy"
    },
    {
      "date": "2026-04-07",
      "high": 22,
      "low": 12,
      "condition": "Snowy"
    },
    {
      "date": "2026-04-08",
      "high": 25,
      "low": 8,
      "condition": "Rainy"
    }
  ],
  "note": "This is mock forecast data for demonstration."
}